In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
df =pd.read_csv("phishing_site_urls.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info

In [ ]:
df.isnull().sum()

In [ ]:
df.Label.value_counts()

In [ ]:
import nltk


In [ ]:
from nltk. tokenize import RegexpTokenizer

In [ ]:
tokenizer = RegexpTokenizer(r'[A-Za-z]+')

In [ ]:
df.URL[0]

In [ ]:
tokenizer.tokenize(df.URL[0])

In [ ]:
df['text_tokenized'] = df.URL.map(lambda t: tokenizer. tokenize(t))

In [ ]:
df.head()

In [ ]:
from nltk.stem. snowball import SnowballStemmer

In [ ]:
stemmer = SnowballStemmer('english')

In [ ]:
df['text_stemmed'] = df['text_tokenized'].map(lambda tokens: [stemmer.stem(word) for word in tokens])

In [ ]:
df.head()

In [ ]:
df['text'] = df['text_stemmed'].map(lambda x: ' '.join(x))

In [ ]:
df.head()

In [ ]:
#Separate good and bad websites
good_sites = df[df['Label'] == 'good']
bad_sites = df[df['Label'] == 'bad']

In [ ]:
good_sites.head()

In [ ]:
bad_sites.head()

In [ ]:
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
import matplotlib.pyplot as plt

def plot_wordcloud(text,
                   mask=None,
                   max_words=400,
                   max_font_size=120,
                   figure_size=(24.0, 16.0),
                   title=None,
                   title_size=40,
                   image_color=False):

    # Stopwords
    stopwords = set(STOPWORDS)
    more_stopwords = {'com', 'http'}
    stopwords = stopwords.union(more_stopwords)

    # Create WordCloud
    wordcloud = WordCloud(
        background_color='white',
        stopwords=stopwords,
        max_words=max_words,
        max_font_size=max_font_size,
        random_state=42,
        mask=mask
    )

    wordcloud.generate(text)

    # Display
    plt.figure(figsize=figure_size)

    if image_color and mask is not None:
        image_colors = ImageColorGenerator(mask)
        plt.imshow(
            wordcloud.recolor(color_func=image_colors),
            interpolation="bilinear"
        )
    else:
        plt.imshow(wordcloud, interpolation="bilinear")

    plt.title(
        title,
        fontdict={
            'size': title_size,
            'color': 'green',
            'verticalalignment': 'bottom'
        }
    )

    plt.axis("off")
    plt.show()


In [ ]:
all_text = ' '.join(good_sites['text'].tolist())

In [ ]:
#Generate word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_text)

In [ ]:
#Display the word cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

In [ ]:
all_text= ''.join(bad_sites['text'].tolist())

#Generate word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(all_text)

#Display the word cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()


In [ ]:
df.head()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
cv = CountVectorizer()
print("scikit-learn installed successfully!")

In [ ]:
features = cv.fit_transform(df['text'])

In [ ]:
features[:5].toarray()

In [ ]:
from sklearn.model_selection._classification_threshold import (
        FixedThresholdClassifier,
         TunedThresholdClassifierCV,
     
      )



In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(features, df.Label)

Model Training


In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
_model = LogisticRegression()

In [ ]:
_model.fit(x_train,y_train)

In [ ]:
_model.score(x_test,y_test)

In [ ]:
_model.score(x_train, y_train)

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print('\nCLASSIFICATION REPORT\n')
print(classification_report(_model.predict(x_test), y_test,
target_names =['Bad', 'Good']))

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
con_mat = pd.DataFrame(confusion_matrix(_model.predict(x_test), y_test),
columns = ['Predicted:Bad', 'Predicted:Good'],
index = ['Actual:Bad', 'Actual:Good'])

In [ ]:
import seaborn as sns



In [ ]:
print('\nCONFUSION MATRIX')
plt.figure(figsize= (6,4))
sns.heatmap(con_mat, annot = True,fmt='d',cmap="YlGnBu")